## **Research Project Section: Data Pre-processing and Management**

### **Overview**
In this phase, the primary focus is to prepare and organize the data by reading and processing large audio recordings (WAV files) and their corresponding annotation files. The process involves extracting relevant spectrogram fragments (representing calls) and storing them in a structured format for further analysis.

### **1. General Data Processing Workflow**

#### **Data Collection**
- Each `*.wav` audio file is paired with its corresponding annotation file `*.Table.1.selections.txt`.
- The annotation file provides detailed information about the number of calls (rows in the file) and their respective time and frequency ranges.

#### **Spectrogram Generation**
- A single large spectrogram is generated for the entire audio file using fixed parameters (e.g., `signal.spectrogram`). The following configurations are applied:
  - **Sampling rate:** Resample audio files to a common rate (e.g., 16 kHz) if necessary.
  - **Spectrogram parameters:** `nperseg`, `nfft`, `noverlap`, and `window`.
- Spectrogram fragments corresponding to each annotated call are extracted.
- Frequency and time constraints (e.g., 20 Hz to 1000 Hz) are optionally applied based on the annotation data.

### **2. Size Standardization**
- A comprehensive analysis of all annotations is performed to identify the maximum call duration and frequency range (i.e., "the longest and widest call").
- To ensure uniformity, zero padding or other alignment techniques (e.g., time stretching) are applied to shorter spectrograms.
- **Objective:** Ensure that all spectrogram fragments have a standardized shape, such as `(F × T)`.

### **3. Creation of "No-call" Fragments**
- Spectrogram fragments representing "no-call" segments are generated by selecting random time intervals that do not overlap with any calls.
- These "no-call" fragments are sized identically to the call spectrograms `(F × T)` and maintain the same frequency range.

### **4. Data Storage**

#### **Saving Processed Data**
- Each extracted spectrogram (call or "no-call") is saved as a 2D array in NumPy format (`.npy` or `.npz`).
- Metadata files, saved in `.csv` format or as Python structures, accompany the spectrogram data and include:
  - **Call type** (e.g., Rupe A, B, Moan, etc.).
  - **Start and end time** within the original audio file.
  - **Original file ID** and any other necessary fields.

### **5. Directory Structure**
The processed data is organized in the following directory structure:
```
**outcome/**
├── processed_calls/ │ 
├── call_001.npy │ 
├── call_002.npy │ 
└── ... ├── processed_no_calls/ │ 
├── no_call_001.npy │ 
├── no_call_002.npy │ 
└── ... 
└── metadata.csv
```
### **6. Additional Considerations**
- Ensure that sample rates and spectrogram dimensions are consistent across all files to avoid mismatches during analysis.
- Implement verification checks after saving the data to ensure file integrity and data accuracy.

By following this structured approach to data pre-processing, the dataset is prepared for efficient and reliable downstream analysis, facilitating accurate research outcomes.

In [2]:
# Import libs
import os
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
import glob
import random

In [3]:
# Define folders path
DATA_FOLDERS = [
    'Rupes A and B',
    'Guttural rupe',
    'Moan',
    'Grey Seal Data Additional'
]

OUTPUT_ROOT = 'outcome'

In [4]:
# Create a folder for the source data, if it does not exist
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [5]:
# Spectrogram parameters
fmin = 20  # Minimum frequency
fmax = 1000  # Maximum frequency
nperseg = 2456  # Number of samples per segment
nfft = 4096  # Number of FFT points
noverlap = 1228  # Overlapping samples
window = 'hann'  # Hann window for spectrogram

# Number of "no-call" segments per audio file
NUM_NO_CALL_SEGMENTS = 5
# Duration (seconds) of each call and no-call segment
SEGMENT_DURATION = 1.0

In [6]:
# Debugging mode
DEBUG = True  # If True, prints additional information

In [7]:
def ensure_mono(samples):
    """
    Ensures that the audio is mono (if stereo, takes the first channel).
    """
    if len(samples.shape) > 1 and samples.shape[1] > 1:
        # If stereo, take the first channel
        return samples[:, 0]
    return samples

In [8]:
def compute_spectrogram(samples, sample_rate):
    """
    Computes the spectrogram for the entire signal.
    Returns: (freqs, times, Sxx).
    """
    freqs, times, Sxx = signal.spectrogram(
        samples,
        fs=sample_rate,
        nperseg=nperseg,
        nfft=nfft,
        noverlap=noverlap,
        window=window
    )
    # Remove very small values
    Sxx[Sxx < 0.001] = 0.001
    return freqs, times, Sxx

In [9]:
def slice_frequency(freqs, Sxx, fmin, fmax):
    """
    Slices the spectrogram by frequency range [fmin, fmax].
    Returns new (freqs, Sxx).
    """
    idx = np.where((freqs >= fmin) & (freqs <= fmax))[0]
    if len(idx) == 0:
        return None, None
    freqs_new = freqs[idx]
    Sxx_new = Sxx[idx, :]
    return freqs_new, Sxx_new

In [10]:
def extract_call(freqs, times, Sxx, t_start, t_end, freq_range=(20, 1000)):
    """
    Extracts a spectrogram segment (Sxx_sub) for the time window [t_start, t_end]
    and frequency range freq_range = (fmin, fmax).
    
    Returns (Sxx_sub, freqs_sub, times_sub) or (None, None, None) if empty.
    """
    time_idx = np.where((times >= t_start) & (times <= t_end))[0]
    freq_idx = np.where((freqs >= freq_range[0]) & (freqs <= freq_range[1]))[0]

    if len(time_idx) == 0 or len(freq_idx) == 0:
        return None, None, None

    Sxx_sub = Sxx[freq_idx][:, time_idx]
    freqs_sub = freqs[freq_idx]
    times_sub = times[time_idx]
    return Sxx_sub, freqs_sub, times_sub

In [11]:
def pad_spectrogram(Sxx_sub, max_freq_points, max_time_points):
    """
    Pads the spectrogram Sxx_sub (shape [freq x time]) to 
    [max_freq_points x max_time_points] by adding zeros.
    """
    freq_dim, time_dim = Sxx_sub.shape

    pad_freq = max_freq_points - freq_dim
    pad_time = max_time_points - time_dim

    if pad_freq < 0 or pad_time < 0:
        # If the segment exceeds max dimensions, trim it
        return Sxx_sub[:max_freq_points, :max_time_points]

    # np.pad for 2D: ((top, bottom), (left, right))
    Sxx_padded = np.pad(
        Sxx_sub,
        pad_width=((0, pad_freq), (0, pad_time)),
        mode='constant',
        constant_values=0.0
    )
    return Sxx_padded

In [12]:
def pick_no_call_segments(annot_df, total_duration, window_length=1.0, n_segments=5, seed=42):
    """
    Returns a list of (start_time, end_time) of length window_length 
    that do not overlap with any 'call' in annot_df.
    """
    random.seed(seed)

    # Collect call intervals
    call_intervals = []
    for _, row in annot_df.iterrows():
        c_start = row['Begin Time (s)']
        c_end = row['End Time (s)']
        call_intervals.append((c_start, c_end))

    call_intervals.sort(key=lambda x: x[0])  # Sort by start time

    no_call_windows = []
    start_all, end_all = total_duration
    attempts = 0
    max_attempts = 1000

    while len(no_call_windows) < n_segments and attempts < max_attempts:
        attempts += 1
        rand_start = random.uniform(start_all, end_all - window_length)
        rand_end = rand_start + window_length

        # Check for overlaps
        overlap = False
        for (cstart, cend) in call_intervals:
            if not (rand_end <= cstart or rand_start >= cend):
                overlap = True
                break

        if not overlap:
            no_call_windows.append((rand_start, rand_end))

    return no_call_windows

In [13]:
# First pass: finding max_freq_points and max_time_points

max_freq_points_global = 0
max_time_points_global = 0

for data_folder in DATA_FOLDERS:
    wav_files = glob.glob(os.path.join(data_folder, '*.wav'))
    for wav_path in wav_files:
        base_name = os.path.splitext(wav_path)[0]
        txt_path = base_name + '.Table.1.selections.txt'
        if not os.path.exists(txt_path):
            if DEBUG:
                print(f"[WARNING] No annotation for {wav_path}")
            continue

        # --- Read audio ---
        sr, samples = wavfile.read(wav_path)
        samples = ensure_mono(samples)  # Convert to mono if needed
        duration_sec = len(samples) / sr
        if DEBUG:
            print(f"\n[DEBUG] File: {os.path.basename(wav_path)}, duration={duration_sec:.2f} s, sr={sr}")

        # --- Compute spectrogram ---
        freqs, times, Sxx = compute_spectrogram(samples, sr)
        freqs, Sxx = slice_frequency(freqs, Sxx, fmin, fmax)
        if freqs is None or Sxx is None:
            continue

        # --- Read annotations ---
        df_annot = pd.read_csv(txt_path, sep='\t')
        for _, row in df_annot.iterrows():
            t_start = row['Begin Time (s)']
            t_end = row['End Time (s)']

            if t_start >= t_end:
                continue

            Sxx_sub, f_sub, t_sub = extract_call(freqs, times, Sxx, t_start, t_end, (fmin, fmax))
            if Sxx_sub is None:
                continue

            freq_dim, time_dim = Sxx_sub.shape
            max_freq_points_global = max(max_freq_points_global, freq_dim)
            max_time_points_global = max(max_time_points_global, time_dim)

print(f"\nGlobal max freq points: {max_freq_points_global}")
print(f"Global max time points: {max_time_points_global}")


[DEBUG] File: 5713.210806110002.wav, duration=1199.98 s, sr=96000

[DEBUG] File: 5713.210807110002.wav, duration=1200.01 s, sr=96000

[DEBUG] File: 5713.210808110002.wav, duration=1200.02 s, sr=96000

[DEBUG] File: 5713.210809120002.wav, duration=1199.98 s, sr=96000

[DEBUG] File: 5713.210811140002.wav, duration=1199.98 s, sr=96000

[DEBUG] File: 5713.210811190002.wav, duration=1199.98 s, sr=96000

[DEBUG] File: 5713.210814160002.wav, duration=1200.01 s, sr=96000

[DEBUG] File: 5713.210823180002.wav, duration=1199.98 s, sr=96000

[DEBUG] File: 5713.210825190002.wav, duration=1200.01 s, sr=96000

[DEBUG] File: 5713.210827200002.wav, duration=1199.95 s, sr=96000

[DEBUG] File: 5713.210908180002.wav, duration=1199.99 s, sr=96000

[DEBUG] File: 5711.211013040024.wav, duration=1199.99 s, sr=96000

[DEBUG] File: 5711.211013050024.wav, duration=1200.01 s, sr=96000

[DEBUG] File: 5711.211015090024.wav, duration=1200.00 s, sr=96000

[DEBUG] File: 5711.211015190024.wav, duration=1200.00 s, sr=9

In [14]:
# Main pass: saving with padding

all_metadata = []

for data_folder in DATA_FOLDERS:
    folder_name = os.path.basename(data_folder)
    output_folder = os.path.join(OUTPUT_ROOT, folder_name)
    os.makedirs(output_folder, exist_ok=True)

    wav_files = glob.glob(os.path.join(data_folder, '*.wav'))
    for wav_path in wav_files:
        base_name = os.path.splitext(wav_path)[0]
        txt_path = base_name + '.Table.1.selections.txt'
        if not os.path.exists(txt_path):
            if DEBUG:
                print(f"[WARNING] No annotation file for {wav_path}")
            continue

        # --- Read audio ---
        sr, samples = wavfile.read(wav_path)
        samples = ensure_mono(samples)
        duration_sec = len(samples) / sr

        # --- Compute spectrogram ---
        freqs, times, Sxx = compute_spectrogram(samples, sr)
        freqs, Sxx = slice_frequency(freqs, Sxx, fmin, fmax)
        if freqs is None or Sxx is None:
            continue

        # --- Read annotations ---
        df_annot = pd.read_csv(txt_path, sep='\t')

        # === (1) Save call spectrograms ===
        for idx_row, row in df_annot.iterrows():
            t_start = row['Begin Time (s)']
            t_end = row['End Time (s)']
            call_type = row.get('Annotation', 'Unknown')

            if t_start >= t_end:
                continue

            Sxx_sub, f_sub, t_sub = extract_call(freqs, times, Sxx, t_start, t_end, (fmin, fmax))
            if Sxx_sub is None:
                continue

            Sxx_padded = pad_spectrogram(Sxx_sub, max_freq_points_global, max_time_points_global)
            output_filename = f"{os.path.basename(base_name)}_call_{idx_row}.npy"
            output_fullpath = os.path.join(output_folder, output_filename)
            np.save(output_fullpath, Sxx_padded)

            meta = {
                'source_wav': os.path.basename(wav_path),
                'annotation_file': os.path.basename(txt_path),
                'call_index': idx_row,
                'call_type': call_type,
                'begin_time': t_start,
                'end_time': t_end,
                'freq_min': fmin,
                'freq_max': fmax,
                'saved_spectrogram': output_filename,
                'label': call_type
            }
            all_metadata.append(meta)

        # === (2) Generate no-call spectrograms ===
        no_call_windows = pick_no_call_segments(
            df_annot,
            total_duration=(0, duration_sec),
            window_length=SEGMENT_DURATION,
            n_segments=NUM_NO_CALL_SEGMENTS,
            seed=42
        )

        for i, (nc_start, nc_end) in enumerate(no_call_windows):
            Sxx_sub, f_sub, t_sub = extract_call(freqs, times, Sxx, nc_start, nc_end, (fmin, fmax))
            if Sxx_sub is None:
                continue

            Sxx_padded = pad_spectrogram(Sxx_sub, max_freq_points_global, max_time_points_global)
            output_filename = f"{os.path.basename(base_name)}_nocall_{i}.npy"
            output_fullpath = os.path.join(output_folder, output_filename)
            np.save(output_fullpath, Sxx_padded)

            meta = {
                'source_wav': os.path.basename(wav_path),
                'annotation_file': os.path.basename(txt_path),
                'call_index': i,
                'call_type': 'no-call',
                'begin_time': nc_start,
                'end_time': nc_end,
                'freq_min': fmin,
                'freq_max': fmax,
                'saved_spectrogram': output_filename,
                'label': 'no-call'
            }
            all_metadata.append(meta)

# Save metadata to CSV
df_meta = pd.DataFrame(all_metadata)
df_meta_path = os.path.join(OUTPUT_ROOT, 'all_metadata.csv')
df_meta.to_csv(df_meta_path, index=False)

print("\n================================================")
print(f"Done. All spectrograms (call & no-call) saved.")
print(f"Metadata in '{df_meta_path}'.")
print("================================================")


Done. All spectrograms (call & no-call) saved.
Metadata in 'outcome\all_metadata.csv'.


## Major changes and improvements Step 1.
-	**Adding functionality for processing different folders with data**: iterate through all folders (Rupes A and B, Guttural rupe, Moan, Gray Seal Data Additional), as well as search for *.wav files through glob.glob. 
    - Purpose: Allows you to automate the processing of the entire dataset, not just one "manual" file.
-	**Check audio if multichannel**: Added ensure_mono(samples) function that selects a single channel if the data is stereo.
    - Purpose: Some spectrogram libraries and methods are designed for a one-dimensional signal. This eliminates possible errors and ensures the uniformity of the format.
-	** Clearing/filling small values in the spectrogram**: Moved to the main function compute_spectrogram(...), applied stably for all files.
    - Purpose: Improves the display of the logarithmic scale, avoiding excessively small quantities that can cause problems during visualization or training.
-	**Cut out call and no-call fragments**: Created a function extract_call(...) that, given [t_start, t_end], highlights a subarray of the spectrogram.
    - Purpose: No-call" is necessary to build models with "call presence vs. no call" recognition. This approach makes it possible to balance or expand the sample, reduce false positives in the detector.
-	**Unification of spectrogram size**: A two-step approach has been created:
    - **First pass** — collection of statistics on the maximum number of frequency bins and time bins for all calls (i.e. max_freq_points_global та max_time_points_global).
    - **Second pass** is a direct cut and completion with zeros (np.pad(...)) so that all spectrograms are the same size.
    - Purpose: Most machine learning models (especially CNNs) require the same size of input data (2D "images"). This simplifies the learning process and greatly improves further analysis.
-	** Saving the results in *.npy and generating metadata **: Each cut spectrogram (call or no-call) is stored as a 2D NumPy array (*.npy).
    - Purpose: The *.npy format allows you to quickly load data without loss to train the model (faster and more convenient than working with images). Metadata helps in analyzing results, debugging, and reproducing experiments.
-	**Improved readability and debugging**: The code is structured by functions (compute_spectrogram, extract_call, pick_no_call_segments, etc.). Added debugging messages ([DEBUG]...) that print key information: how long the audio lasts, what speakers are in the annotations, which returns extract_call.
    - Purpose: This makes the project easier to understand and scale, and makes it easier to find errors or analyze why certain spectrograms are cut/not cut.